In [2]:
"""
CoolProp Calculator
A comprehensive tool for thermophysical property calculations with multiple interfaces
"""

import CoolProp.CoolProp as cp
import json
import csv
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
import sys


class CoolPropCalculator:
    """CoolProp calculator with multiple calculation modes"""
    
    # Property definitions
    PROPERTIES = {
        'T': {'name': 'Temperature', 'unit': 'K', 'description': 'Absolute temperature'},
        'P': {'name': 'Pressure', 'unit': 'Pa', 'description': 'Absolute pressure'},
        'H': {'name': 'Enthalpy', 'unit': 'J/kg', 'description': 'Specific enthalpy'},
        'S': {'name': 'Entropy', 'unit': 'J/kg·K', 'description': 'Specific entropy'},
        'D': {'name': 'Density', 'unit': 'kg/m³', 'description': 'Mass density'},
        'Q': {'name': 'Quality', 'unit': '-', 'description': 'Vapor quality (0=liquid, 1=vapor)'},
        'V': {'name': 'Specific Volume', 'unit': 'm³/kg', 'description': 'Specific volume'},
        'U': {'name': 'Internal Energy', 'unit': 'J/kg', 'description': 'Specific internal energy'},
        'G': {'name': 'Gibbs Energy', 'unit': 'J/kg', 'description': 'Specific Gibbs free energy'},
        'A': {'name': 'Helmholtz Energy', 'unit': 'J/kg', 'description': 'Specific Helmholtz free energy'},
        'viscosity': {'name': 'Viscosity', 'unit': 'Pa·s', 'description': 'Dynamic viscosity'},
        'conductivity': {'name': 'Thermal Conductivity', 'unit': 'W/m·K', 'description': 'Thermal conductivity'},
        'Prandtl': {'name': 'Prandtl Number', 'unit': '-', 'description': 'Prandtl number'},
        'surface_tension': {'name': 'Surface Tension', 'unit': 'N/m', 'description': 'Surface tension'},
        'speed_sound': {'name': 'Speed of Sound', 'unit': 'm/s', 'description': 'Speed of sound'},
        'C': {'name': 'Specific Heat (const P)', 'unit': 'J/kg·K', 'description': 'Specific heat at constant pressure'},
        'CVMASS': {'name': 'Specific Heat (const V)', 'unit': 'J/kg·K', 'description': 'Specific heat at constant volume'},
        'isentropic_expansion_coefficient': {'name': 'Isentropic Exp. Coeff.', 'unit': '-', 'description': 'Isentropic expansion coefficient'},
        'isothermal_compressibility': {'name': 'Isothermal Compress.', 'unit': '1/Pa', 'description': 'Isothermal compressibility'},
    }
    
    # Critical properties
    CRITICAL_PROPERTIES = {
        'T_critical': {'name': 'Critical Temperature', 'unit': 'K'},
        'P_critical': {'name': 'Critical Pressure', 'unit': 'Pa'},
        'rhocrit': {'name': 'Critical Density', 'unit': 'kg/m³'},
    }
    
    # Triple point properties
    TRIPLE_PROPERTIES = {
        'T_triple': {'name': 'Triple Point Temperature', 'unit': 'K'},
        'p_triple': {'name': 'Triple Point Pressure', 'unit': 'Pa'},
    }
    
    # Complete list of substances available in CoolProp
    COMMON_SUBSTANCES = [
        # Pure fluids
        'Water', 'Air', 'Nitrogen', 'Oxygen', 'Hydrogen', 'Helium', 'Argon', 'Neon', 'Krypton', 'Xenon',
        
        # Hydrocarbons
        'Methane', 'Ethane', 'Propane', 'n-Butane', 'IsoButane', 'n-Pentane', 'Isopentane', 'n-Hexane',
        'n-Heptane', 'n-Octane', 'n-Nonane', 'n-Decane', 'Ethylene', 'Propylene', 'Acetone',
        
        # Refrigerants - HFCs
        'R134a', 'R32', 'R125', 'R143a', 'R152a', 'R227ea', 'R236fa', 'R245fa', 'R365mfc',
        
        # Refrigerants - HFOs
        'R1234yf', 'R1234ze(E)', 'R1233zd(E)', 'R1336mzz(Z)',
        
        # Refrigerant Blends
        'R404A', 'R407C', 'R410A', 'R507A', 'R407F', 'R417A', 'R422D', 'R424A', 'R426A', 'R427A',
        'R428A', 'R434A', 'R437A', 'R438A', 'R448A', 'R449A', 'R450A', 'R452A', 'R513A',
        
        # Natural refrigerants
        'Ammonia', 'CarbonDioxide', 'R290', 'R600', 'R600a', 'R717', 'R744', 'R1270',
        
        # Alcohols
        'Methanol', 'Ethanol', 'Propanol', 'IsoButanol',
        
        # Aromatics
        'Benzene', 'Toluene', 'Xylene', 'EthylBenzene', 'CycloHexane',
        
        # Other compounds
        'SulfurDioxide', 'SulfurHexafluoride', 'CarbonMonoxide', 'NitrousOxide',
        'Acetylene', 'DimethylEther', 'DimethylCarbonate',
        
        # Siloxanes
        'D4', 'D5', 'D6', 'MD2M', 'MD3M', 'MD4M', 'MDM', 'MM',
        
        # Heat transfer fluids
        'HFE143m', 'HFE7000', 'HFE7100', 'HFE7200', 'HFE7300', 'HFE7500',
        'Novec649', 'RC318',
        
        # Cryogenics
        'ParaHydrogen', 'OrthoHydrogen', 'Deuterium',
    ]
    
    def __init__(self):
        self.calculation_history = []
    
    def calculate_property(self, output_prop: str, input1_prop: str, input1_value: float,
                          input2_prop: str, input2_value: float, substance: str) -> Dict[str, Any]:
        """
        Calculate a thermophysical property
        
        Returns:
            Dictionary with result, metadata, and status
        """
        try:
            result = cp.PropsSI(output_prop, input1_prop, input1_value, 
                              input2_prop, input2_value, substance)
            
            calculation = {
                'timestamp': datetime.now().isoformat(),
                'substance': substance,
                'output_property': output_prop,
                'output_value': result,
                'input1_property': input1_prop,
                'input1_value': input1_value,
                'input2_property': input2_prop,
                'input2_value': input2_value,
                'status': 'success'
            }
            
            self.calculation_history.append(calculation)
            return calculation
            
        except Exception as e:
            error_calc = {
                'timestamp': datetime.now().isoformat(),
                'substance': substance,
                'output_property': output_prop,
                'error': str(e),
                'status': 'error'
            }
            self.calculation_history.append(error_calc)
            return error_calc
    
    def get_critical_properties(self, substance: str) -> Dict[str, float]:
        """Get all critical properties for a substance"""
        try:
            return {
                'T_critical': cp.PropsSI('Tcrit', substance),
                'P_critical': cp.PropsSI('pcrit', substance),
                'rhocrit': cp.PropsSI('rhocrit', substance),
            }
        except Exception as e:
            return {'error': str(e)}
    
    def get_triple_properties(self, substance: str) -> Dict[str, float]:
        """Get triple point properties for a substance"""
        try:
            return {
                'T_triple': cp.PropsSI('Ttriple', substance),
                'p_triple': cp.PropsSI('ptriple', substance),
            }
        except Exception as e:
            return {'error': str(e)}
    
    def get_all_properties(self, input1_prop: str, input1_value: float,
                          input2_prop: str, input2_value: float, 
                          substance: str) -> Dict[str, Any]:
        """Calculate all available properties at once"""
        results = {
            'substance': substance,
            'inputs': {
                input1_prop: input1_value,
                input2_prop: input2_value
            },
            'properties': {}
        }
        
        for prop_key in self.PROPERTIES.keys():
            if prop_key not in [input1_prop, input2_prop]:
                try:
                    value = cp.PropsSI(prop_key, input1_prop, input1_value,
                                     input2_prop, input2_value, substance)
                    results['properties'][prop_key] = {
                        'value': value,
                        'unit': self.PROPERTIES[prop_key]['unit'],
                        'name': self.PROPERTIES[prop_key]['name']
                    }
                except:
                    pass
        
        return results
    
    def batch_calculate(self, calculations: List[Dict]) -> List[Dict]:
        """
        Process multiple calculations at once
        
        Args:
            calculations: List of calculation dictionaries with keys:
                         output_prop, input1_prop, input1_value, input2_prop, input2_value, substance
        """
        results = []
        for calc in calculations:
            result = self.calculate_property(**calc)
            results.append(result)
        return results
    
    def create_state_point_table(self, substance: str, temps: List[float], 
                                pressures: List[float]) -> List[Dict]:
        """Create a table of properties at different state points"""
        table = []
        for T in temps:
            for P in pressures:
                try:
                    state = {
                        'T': T,
                        'P': P,
                        'H': cp.PropsSI('H', 'T', T, 'P', P, substance),
                        'S': cp.PropsSI('S', 'T', T, 'P', P, substance),
                        'D': cp.PropsSI('D', 'T', T, 'P', P, substance),
                        'viscosity': cp.PropsSI('V', 'T', T, 'P', P, substance),
                        'conductivity': cp.PropsSI('L', 'T', T, 'P', P, substance),
                    }
                    table.append(state)
                except:
                    pass
        return table


class InteractiveCLI:
    """Enhanced interactive command-line interface"""
    
    def __init__(self):
        self.calc = CoolPropCalculator()
        self.running = True
    
    def display_menu(self):
        """Display main menu"""
        print("\n" + "="*60)
        print("  ENHANCED COOLPROP CALCULATOR")
        print("="*60)
        print("\n[1] Single Property Calculation")
        print("[2] All Properties at State Point")
        print("[3] Critical Properties")
        print("[4] Triple Point Properties")
        print("[5] Batch Calculations")
        print("[6] Property Table Generator")
        print("[7] List Available Substances")
        print("[8] Property Reference Guide")
        print("[0] Exit")
        print("\n" + "-"*60)
    
    def single_calculation(self):
        """Interactive single property calculation"""
        print("\n--- Single Property Calculation ---\n")
        
        # Show available properties
        print("Available properties:")
        for i, (key, val) in enumerate(self.calc.PROPERTIES.items(), 1):
            print(f"  {key:15} - {val['name']} [{val['unit']}]")
        
        output_prop = input("\nProperty to calculate: ").strip()
        if output_prop not in self.calc.PROPERTIES:
            print("Invalid property!")
            return
        
        substance = input("Substance: ").strip()
        
        print("\nFirst input property:")
        input1_prop = input("Property: ").strip()
        if input1_prop not in self.calc.PROPERTIES:
            print("Invalid property!")
            return
        input1_value = float(input(f"Value [{self.calc.PROPERTIES[input1_prop]['unit']}]: "))
        
        print("\nSecond input property:")
        input2_prop = input("Property: ").strip()
        if input2_prop not in self.calc.PROPERTIES:
            print("Invalid property!")
            return
        input2_value = float(input(f"Value [{self.calc.PROPERTIES[input2_prop]['unit']}]: "))
        
        result = self.calc.calculate_property(output_prop, input1_prop, input1_value,
                                             input2_prop, input2_value, substance)
        
        if result['status'] == 'success':
            print(f"\n{'='*60}")
            print(f"Result: {self.calc.PROPERTIES[output_prop]['name']}")
            print(f"Value:  {result['output_value']:.12e} {self.calc.PROPERTIES[output_prop]['unit']}")
            print(f"{'='*60}")
        else:
            print(f"\nError: {result['error']}")
    
    def all_properties(self):
        """Calculate all properties at a state point"""
        print("\n--- All Properties at State Point ---\n")
        
        substance = input("Substance: ").strip()
        
        print("\nFirst input property:")
        input1_prop = input("Property (e.g., T, P): ").strip()
        input1_value = float(input(f"Value [{self.calc.PROPERTIES.get(input1_prop, {'unit': '?'})['unit']}]: "))
        
        print("\nSecond input property:")
        input2_prop = input("Property (e.g., P, D): ").strip()
        input2_value = float(input(f"Value [{self.calc.PROPERTIES.get(input2_prop, {'unit': '?'})['unit']}]: "))
        
        results = self.calc.get_all_properties(input1_prop, input1_value,
                                              input2_prop, input2_value, substance)
        
        print(f"\n{'='*70}")
        print(f"All Properties for {substance}")
        print(f"At {input1_prop}={input1_value}, {input2_prop}={input2_value}")
        print(f"{'='*70}\n")
        
        for prop, data in results['properties'].items():
            print(f"{data['name']:30} = {data['value']:20.12e} {data['unit']}")
    
    def critical_properties(self):
        """Display critical properties"""
        print("\n--- Critical Properties ---\n")
        substance = input("Substance: ").strip()
        
        props = self.calc.get_critical_properties(substance)
        
        if 'error' not in props:
            print(f"\nCritical Properties for {substance}:")
            print(f"  Temperature: {props['T_critical']:.10f} K")
            print(f"  Pressure:    {props['P_critical']:.12e} Pa")
            print(f"  Density:     {props['rhocrit']:.10f} kg/m³")
        else:
            print(f"Error: {props['error']}")
    
    def batch_calculations(self):
        """Interactive batch calculations"""
        print("\n--- Batch Calculations ---\n")
        print("Enter multiple calculations (type 'done' when finished)\n")
        
        calculations = []
        calc_num = 1
        
        while True:
            print(f"\nCalculation #{calc_num}:")
            substance = input("  Substance (or 'done' to finish): ").strip()
            
            if substance.lower() == 'done':
                break
            
            try:
                output_prop = input("  Output property: ").strip()
                input1_prop = input("  First input property: ").strip()
                input1_value = float(input("  First input value: "))
                input2_prop = input("  Second input property: ").strip()
                input2_value = float(input("  Second input value: "))
                
                calculations.append({
                    'output_prop': output_prop,
                    'input1_prop': input1_prop,
                    'input1_value': input1_value,
                    'input2_prop': input2_prop,
                    'input2_value': input2_value,
                    'substance': substance
                })
                calc_num += 1
            except ValueError:
                print("Invalid input! Skipping this calculation.")
                continue
        
        if calculations:
            print(f"\n{'='*70}")
            print(f"Processing {len(calculations)} calculations...")
            print(f"{'='*70}\n")
            
            results = self.calc.batch_calculate(calculations)
            
            for i, result in enumerate(results, 1):
                if result['status'] == 'success':
                    print(f"{i}. {result['substance']}: {result['output_property']} = {result['output_value']:.12e}")
                else:
                    print(f"{i}. ERROR - {result.get('error', 'Unknown error')}")
        else:
            print("\nNo calculations to process.")
    
    def property_table_generator(self):
        """Generate a property table"""
        print("\n--- Property Table Generator ---\n")
        
        substance = input("Substance: ").strip()
        
        print("\nTemperature range:")
        t_start = float(input("  Start (K): "))
        t_end = float(input("  End (K): "))
        t_step = float(input("  Step (K): "))
        
        print("\nPressure range:")
        p_start = float(input("  Start (Pa): "))
        p_end = float(input("  End (Pa): "))
        p_step = float(input("  Step (Pa): "))
        
        # Generate ranges
        temps = []
        T = t_start
        while T <= t_end:
            temps.append(T)
            T += t_step
        
        pressures = []
        P = p_start
        while P <= p_end:
            pressures.append(P)
            P += p_step
        
        print(f"\nGenerating table with {len(temps)} temperatures x {len(pressures)} pressures = {len(temps) * len(pressures)} points...")
        
        table = self.calc.create_state_point_table(substance, temps, pressures)
        
        print(f"\n{'='*120}")
        print(f"Property Table for {substance}")
        print(f"{'='*120}")
        print(f"{'T (K)':>15} {'P (Pa)':>20} {'H (J/kg)':>20} {'S (J/kg·K)':>20} {'D (kg/m³)':>20}")
        print("-" * 120)
        
        for state in table[:20]:  # Show first 20 rows
            print(f"{state['T']:15.8f} {state['P']:20.12e} {state['H']:20.12e} {state['S']:20.12e} {state['D']:20.12e}")
        
        if len(table) > 20:
            print(f"\n... ({len(table) - 20} more rows) ...")
        
        # Offer to save
        save = input("\nSave table to CSV? (y/n): ").strip().lower()
        if save == 'y':
            filename = input("Filename: ").strip()
            if not filename.endswith('.csv'):
                filename += '.csv'
            
            with open(filename, 'w', newline='') as f:
                if table:
                    writer = csv.DictWriter(f, fieldnames=table[0].keys())
                    writer.writeheader()
                    writer.writerows(table)
            print(f"Table saved to {filename}")
    
    def list_substances(self):
        """List all available substances"""
        print("\n--- Available Substances ---\n")
        print(f"Total: {len(self.calc.COMMON_SUBSTANCES)} substances\n")
        
        # Display in columns
        substances = sorted(self.calc.COMMON_SUBSTANCES)
        col_width = 25
        cols = 3
        
        for i in range(0, len(substances), cols):
            row = substances[i:i+cols]
            print("".join(f"{s:<{col_width}}" for s in row))
    
    def property_guide(self):
        """Display property reference guide"""
        print("\n" + "="*70)
        print("PROPERTY REFERENCE GUIDE")
        print("="*70 + "\n")
        
        for key, val in self.calc.PROPERTIES.items():
            print(f"{key:15} | {val['name']:25} | {val['unit']:12} | {val['description']}")
    
    def run(self):
        """Run the interactive CLI"""
        while self.running:
            self.display_menu()
            choice = input("\nSelect option: ").strip()
            
            try:
                if choice == '1':
                    self.single_calculation()
                elif choice == '2':
                    self.all_properties()
                elif choice == '3':
                    self.critical_properties()
                elif choice == '4':
                    substance = input("\nSubstance: ").strip()
                    props = self.calc.get_triple_properties(substance)
                    if 'error' not in props:
                        print(f"\nTriple Point for {substance}:")
                        print(f"  Temperature: {props['T_triple']:.10f} K")
                        print(f"  Pressure:    {props['p_triple']:.12e} Pa")
                    else:
                        print(f"Error: {props['error']}")
                elif choice == '5':
                    self.batch_calculations()
                elif choice == '6':
                    self.property_table_generator()
                elif choice == '7':
                    self.list_substances()
                elif choice == '8':
                    self.property_guide()
                elif choice == '0':
                    print("\nGoodbye!")
                    self.running = False
                else:
                    print("Invalid option!")
            except KeyboardInterrupt:
                print("\n\nOperation cancelled.")
            except Exception as e:
                print(f"\nError: {e}")


def main():
    """Main entry point"""
    if len(sys.argv) > 1 and sys.argv[1] == '--simple':
        # Simple mode (original style)
        calc = CoolPropCalculator()
        print("\nCOOLPROP CALCULATOR (Simple Mode)\n")
        # ... original simple mode code ...
    else:
        # Enhanced interactive mode
        cli = InteractiveCLI()
        cli.run()


if __name__ == "__main__":
    main()


  ENHANCED COOLPROP CALCULATOR

[1] Single Property Calculation
[2] All Properties at State Point
[3] Critical Properties
[4] Triple Point Properties
[5] Batch Calculations
[6] Property Table Generator
[7] List Available Substances
[8] Property Reference Guide
[0] Exit

------------------------------------------------------------

Select option: 2

--- All Properties at State Point ---

Substance: Water

First input property:
Property (e.g., T, P): T
Value [K]: 298.15

Second input property:
Property (e.g., P, D): P
Value [Pa]: 101325

All Properties for Water
At T=298.15, P=101325.0

Enthalpy                       =   1.049201198094e+05 J/kg
Entropy                        =   3.671996421056e+02 J/kg·K
Density                        =   9.970476367603e+02 kg/m³
Quality                        =  -1.000000000000e+00 -
Specific Volume                =   8.900224890777e-04 m³/kg
Internal Energy                =   1.048184947753e+05 J/kg
Gibbs Energy                   =  -4.560453484421e+0